In [ ]:
import datetime
import os
from glob import glob

import geopandas as gpd
import netCDF4 as nc
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.dates import YearLocator, DateFormatter
from matplotlib.ticker import MultipleLocator

### Load data

In [ ]:
data_path = 'ME4OH_EN411_OFAM3/data/en4.1.1/1993-2014/'
files = [f for f in glob(os.path.join(data_path, "*.nc"))]

In [ ]:
file2read = nc.Dataset(files[0],'r')
print(file2read.variables.keys())

In [ ]:
def load_and_filter_ohc(filepath):
    """Load data file, extract variables, filter for layer 1 OHC"""
    file_data = nc.Dataset(filepath,'r')

    # Extract variables
    lat = file_data["ts_lat"][:].data  # Latitude
    long = file_data["ts_lon"][:].data  # Longitude
    time = file_data["en4_ymd"][:].data  # Year , month , day
    mask = file_data["dohc_mask_by_en4_maxdepth"][:].data # Valid profile mask
    dohc1 = file_data["dohc"][:, 0].data  # Layer 1 OHC anomaly target (NOTE: index 0 in python, index 1 in R)
    ssh = file_data["eta_t"][:].data  # Sea surface height
    wmo_inst_type = file_data["en4_wmo_inst_type"][:].data
    proj_name = file_data["en4_project_name"][:].data
    plat_num = file_data["en4_platform_number"][:].data

    # Filter to valid OHC
    # Only valid Layer-1 profiles included according to mask and remove NaNs
    idx = [i for i in range(len(mask)) if mask[i, 0] == 1]
    filtered_dohc1 = np.array([dohc1[i] for i in idx])
    filtered_lat = np.array([lat[i] for i in idx])
    filtered_long = np.array([long[i] for i in idx])
    filtered_date = np.array([datetime.date(int(time[i, 0]), int(time[i, 1]), int(time[i, 2])) for i in idx])
    filtered_ssh = np.array([ssh[i] for i in idx])
    filtered_wmo = np.array([''.join(list(map(lambda x: x.decode('utf-8'), wmo_inst_type[i]))).strip() for i in idx])
    filtered_proj_name = np.array([''.join(list(map(lambda x: x.decode('utf-8'), proj_name[i]))).strip() for i in idx])
    filtered_plat_num = np.array([''.join(list(map(lambda x: x.decode('utf-8'), plat_num[i]))).strip() for i in idx])

    # Make df
    ohc_df = pd.DataFrame({
        "Date": filtered_date,
        "Latitude": filtered_lat,
        "Longitude": filtered_long,
        "OHC": filtered_dohc1,
        "SSH": filtered_ssh,
        "WMO Instrument Type": filtered_wmo,
        "Project name": filtered_proj_name,
        "Platform number": filtered_plat_num
    })
    ohc_df = ohc_df.dropna()
    
    return ohc_df
    

In [ ]:
# Read all files
ohc_df = load_and_filter_ohc(files[0])
for path in files[1:]:
    file_df = load_and_filter_ohc(path)
    ohc_df = pd.concat([ohc_df, file_df])
ohc_df['Date'] = pd.to_datetime(ohc_df['Date'])
ohc_df

In [ ]:
# Convert longitude to -180 to 180 format, allows plotting against geopandas world maps
ohc_df['Longitude'] = ohc_df['Longitude'].apply(lambda x: -(360-x) if x > 180 else x)
ohc_df

In [ ]:
# Filter by Argo platforms
argo_df = ohc_df[ohc_df['Project name'].isin(['ARGOR', 'ARGOD', 'ARGOA'])].sort_values('Date').reset_index(drop=True)
argo_df

# EDA

- Amount of data points across time
- Amount of data across space
- Plot OHC vs lat long for a single month?

In [ ]:
sept_93 = ohc_df.loc[(ohc_df['Date'].dt.month == 9) & (ohc_df['Date'].dt.year == 1993)]
sept_93

In [ ]:
df = ohc_df.copy(deep=True)
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df.sort_values('Date').reset_index(drop=True)

### Monthly mean OHC

In [ ]:
monthly_means = df.groupby(df['Date'].dt.strftime('%m/%Y'))['OHC'].mean().sort_values()
monthly_std = df.groupby(df['Date'].dt.strftime('%m/%Y'))['OHC'].std().sort_values()

In [ ]:
df_monthly = pd.DataFrame(monthly_means).reset_index()
df_monthly['Date'] = pd.to_datetime(df_monthly['Date'], format="%m/%Y")
df_monthly["std"] = monthly_std
df_monthly = df_monthly.sort_values('Date').reset_index(drop=True)
df_monthly


In [ ]:
# sns.lineplot(df_monthly, x='Date', y='OHC')
plt.plot(df_monthly['Date'], df_monthly['OHC'])
plt.ylabel("Mean OHC across all measurements (per month)")
plt.xlabel("Year")

### Monthly counts

In [ ]:
df['year_month'] = df['Date'].dt.to_period('M')
monthly_counts = df.groupby('year_month').size()

df_counts = pd.DataFrame(monthly_counts).reset_index()
df_counts.rename(columns={0: 'Count', 'year_month': 'Month'}, inplace=True)
df_counts['Month'] = pd.to_datetime(df_counts['Month'].astype(str), format="%Y-%m")
df_counts

In [ ]:
argo_df['year_month'] = argo_df['Date'].dt.to_period('M')
monthly_argo_counts = argo_df.groupby('year_month').size()

df_argo_counts = pd.DataFrame(monthly_argo_counts).reset_index()
df_argo_counts.rename(columns={0: 'Count', 'year_month': 'Month'}, inplace=True)
df_argo_counts['Month'] = pd.to_datetime(df_argo_counts['Month'].astype(str), format="%Y-%m")
df_argo_counts

In [ ]:
month_list = pd.date_range('1993-1-1','2000-1-1', freq='MS').strftime('%m/%Y').tolist()
df_zeros = pd.DataFrame({
    "Month": month_list,
    "Count": np.zeros(len(month_list)),
})
df_zeros['Month'] = pd.to_datetime(df_zeros['Month'], format="%m/%Y")
df_zeros

In [ ]:
# plt.figure(figsize=(12,6))
plt.plot(df_counts['Month'], df_counts['Count'], marker='.')
plt.grid(True)
plt.ylabel("Number of observations per month")
plt.xlabel("Year")
plt.axvline(x=datetime.datetime(2000, 1, 1), c='red', ls='--')
ax = plt.gca()
ax.set_xlim([datetime.datetime(1993,1,1), datetime.datetime(2015,1,1)])

ax.xaxis.set_major_locator(YearLocator(1))
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.xticks(rotation=90)

plt.ticklabel_format(axis='y', style='sci', scilimits=(3,3))


In [ ]:
comp_df = pd.DataFrame()
comp_df['Month'] = pd.date_range(start = '1993-01-01', end = '2014-12-01', freq='MS')
comp_df['Month'] = pd.to_datetime(comp_df['Month'])
comp_df['Argo'] = pd.concat([df_zeros, pd.DataFrame(df_argo_counts)], ignore_index=True)['Count']
comp_df['Other'] = pd.DataFrame(df_counts)['Count'] - pd.concat([df_zeros, pd.DataFrame(df_argo_counts)], ignore_index=True)['Count']
comp_df

In [ ]:
comp_df_plot = comp_df.set_index('Month')
comp_df_plot.index = comp_df_plot.index.to_period('M').year  # Using period indexing from pandas, without this get an error
ax = comp_df_plot.plot(kind='bar', stacked=True, figsize=(12, 6), width=0.6)
ax.set_ylabel('Number of observations per month')
ax.set_xlabel('Year')
ax.set_title('Monthly observation counts: Argo vs Other platforms')

ax.xaxis.set_major_locator(MultipleLocator(12))  # Have stored years only as the period index, so shows only year
# If you want to plot by month, remove .year from comp_df_plot.index.to_period('M').year above and then will show months and years
plt.ticklabel_format(axis='y', style='sci', scilimits=(3,3))

plt.tight_layout()
plt.legend(title="Platform type")

In [ ]:
ohc_df["Platform number"].unique()

In [ ]:
ohc_df["Project name"].unique()

In [ ]:
# ohc_df.plot.scatter(x='Date', y='OHC', c='Platform number')

In [ ]:
platform_counts = df.groupby(['Project name', 'Platform number']).size()

df_counts = pd.DataFrame(platform_counts).reset_index()
df_counts.rename(columns={0: 'Count'}, inplace=True)
df_counts[df_counts['Platform number'].duplicated(keep=False)].to_csv('duplcated_platforms.csv')


In [ ]:
df['Unique ID'] = df['Project name'] + ' ' + df['Platform number']

## Plot OHC in latitude and longitude
Plot for each month

In [ ]:
df

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
# ocean_url = "https://naciscdn.org/naturalearth/110m/physical/ne_110m_ocean.zip"
world = gpd.read_file(url).rename({"ADMIN": "name"}, axis="columns")
# oceans = gpd.read_file(ocean_url)

In [ ]:
month_list = pd.date_range('1993-1-1','2014-12-31', freq='MS').strftime('%Y-%m').tolist()
num_plots = 10
plot_months = [month_list[i] for i in range(0, len(month_list), len(month_list)//num_plots)]

In [ ]:
for n,g in df.groupby('year_month'):
    if str(n) in plot_months: 
        fig, ax = plt.subplots(figsize=(15, 9))
        world.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries

        # Create a shared normalization object
        norm = matplotlib.colors.Normalize(vmin=min(df['OHC']), vmax=max(df['OHC']))
        # Define the colormap
        cmap = plt.cm.plasma

        s = ax.scatter(g.Longitude, g.Latitude, c=g.OHC, s=1, cmap='plasma')  # Plot OHC observations
        cbar = plt.colorbar(s, ax=ax, cmap=cmap, norm=norm)

        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_facecolor("lightblue")

        plt.title(n)
        plt.show()
        plt.close()

In [ ]:
df['year'] = df['Date'].dt.to_period('Y')

In [ ]:
for n,g in df.groupby('year'):
    fig, ax = plt.subplots(figsize=(15, 9))
    world.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries

    # Create a shared normalization object
    norm = matplotlib.colors.Normalize(vmin=min(df['OHC']), vmax=max(df['OHC']))
    # Define the colormap
    cmap = plt.cm.plasma

    s = ax.scatter(g.Longitude, g.Latitude, c=g.OHC, s=1, cmap='plasma')  # Plot OHC observations
    cbar = plt.colorbar(s, ax=ax, cmap=cmap, norm=norm)

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_facecolor("lightblue")
    plt.title(n)

    plt.show()
    plt.close()